# MCP Server サンプル (FastMCP) on Google Colab

公式Python SDK (`mcp`) の **FastMCP** を使ったサンプルです。
上から順にセルを実行してください。

| Step | 内容 |
|---|---|
| 1 | パッケージインストール |
| 2 | server.py の書き出し |
| 3 | server.py の確認 |
| 4 | インポートとテスト |
| 5 | MCPクライアントで接続テスト |


## Step 1: パッケージのインストール


In [ ]:
!pip install 'mcp[cli]' nest_asyncio -q
import nest_asyncio
nest_asyncio.apply()
print('インストール完了')


## Step 2: server.py の書き出し

FastMCPを使ったサーバーを `/content/server.py` に書き出します。


In [ ]:
# ✅ Step 1: server.py を /content/server.py に書き出す
import os

server_lines = [
    "from mcp.server.fastmcp import FastMCP\n",
    "import random\n",
    "\n",
    "# FastMCP インスタンスを作成\n",
    "mcp = FastMCP('colab-demo')\n",
    "\n",
    "\n",
    "# ── Tool 1: 足し算 ──────────────────────────────────────\n",
    "@mcp.tool()\n",
    "def add(a: int, b: int) -> int:\n",
    "    '''2つの整数を足し算して返す'''\n",
    "    return a + b\n",
    "\n",
    "\n",
    "# ── Tool 2: 挨拶 ────────────────────────────────────────\n",
    "@mcp.tool()\n",
    "def greet(name: str) -> str:\n",
    "    '''名前を受け取って日本語で挨拶を返す'''\n",
    "    return f'こんにちは、{name}さん！MCPサーバーへようこそ'\n",
    "\n",
    "\n",
    "# ── Tool 3: ダミー天気 ───────────────────────────────────\n",
    "@mcp.tool()\n",
    "def get_weather(city: str) -> str:\n",
    "    '''指定した都市のダミー天気情報を返す'''\n",
    "    conditions = ['晴れ', '曇り', '雨', '雪']\n",
    "    temp = random.randint(0, 35)\n",
    "    cond = random.choice(conditions)\n",
    "    return f'{city}の天気: {cond}, 気温: {temp}度C (デモデータ)'\n",
    "\n",
    "\n",
    "# ── Resource ─────────────────────────────────────────────\n",
    "@mcp.resource('memo://sample')\n",
    "def get_memo() -> str:\n",
    "    '''サンプルメモリソース'''\n",
    "    return 'これはMCPサーバーのサンプルリソースです。\\nGoogle Colab上で動作しています。'\n",
    "\n",
    "\n",
    "# ── Prompt ───────────────────────────────────────────────\n",
    "@mcp.prompt()\n",
    "def summarize(text: str) -> str:\n",
    "    '''テキストを要約するプロンプトテンプレート'''\n",
    "    return f'次のテキストを3行以内で要約してください:\\n\\n{text}'\n",
]

with open('/content/server.py', 'w', encoding='utf-8') as f:
    f.writelines(server_lines)

size = os.path.getsize('/content/server.py')
print(f'server.py を書き出しました ({size} bytes)')


## Step 3: server.py の内容確認


In [ ]:
# ファイルの存在確認
import os
path = '/content/server.py'
assert os.path.exists(path), 'server.py が見つかりません。Step 2を先に実行してください'
print(f'OK: {path} ({os.path.getsize(path)} bytes)')
print()
with open(path) as f:
    print(f.read())


## Step 4: サーバーのインポートと動作テスト

FastMCPではハンドラ関数を直接テストできます。


In [ ]:
import sys
sys.path.insert(0, '/content')

# キャッシュを削除して確実に再ロード
if 'server' in sys.modules:
    del sys.modules['server']

import server
print('server.py のインポート成功')
print('登録されているツール:', [t.name for t in server.mcp._tool_manager.list_tools()])


### ツールの直接呼び出しテスト


In [ ]:
# add ツール
result = server.add(42, 58)
print('add(42, 58) =', result)


In [ ]:
# greet ツール
result = server.greet('Claude')
print('greet:', result)


In [ ]:
# get_weather ツール
result = server.get_weather('Tokyo')
print('get_weather:', result)


In [ ]:
# resource
result = server.get_memo()
print('memo resource:')
print(result)


In [ ]:
# prompt template
result = server.summarize('MCPはAnthropicが開発したオープンプロトコルです。AIと外部ツールを標準的な方法で接続します。')
print('summarize prompt:')
print(result)


## Step 5: MCPクライアントで接続テスト

公式SDKの `ClientSession` を使って実際のMCPプロトコルで通信します。

> **注意**: ColabのJupyterカーネルは標準入出力をラップしているため `fileno()` が使えず、
> stdioトランスポートは動作しません。
> 代わりに `create_connected_server_and_client_session` でインプロセス接続します。
> JSON-RPCのシリアライズ・デシリアライズは実際に行われるため、本物のMCPプロトコル通信です。


In [ ]:
import asyncio
from mcp.server.fastmcp import FastMCP
from mcp.shared.memory import (
    create_connected_server_and_client_session as create_session,
)

async def run_client():
    # インプロセスでサーバーとクライアントを接続
    # (Colabはstdioのfileno()が使えないためインプロセス方式を使用)
    async with create_session(server.mcp._mcp_server) as client_session:
        print('=== サーバー接続成功 (in-process) ===')

        # ツール一覧
        tools = await client_session.list_tools()
        print('\n登録ツール一覧:')
        for t in tools.tools:
            print(f'  {t.name}: {t.description}')

        # add ツール呼び出し
        res = await client_session.call_tool('add', {'a': 10, 'b': 32})
        print('\nadd(10, 32) =', res.content[0].text)

        # greet ツール呼び出し
        res = await client_session.call_tool('greet', {'name': 'World'})
        print('greet:', res.content[0].text)

        # get_weather ツール呼び出し
        res = await client_session.call_tool('get_weather', {'city': 'Tokyo'})
        print('get_weather:', res.content[0].text)

        # リソース一覧
        resources = await client_session.list_resources()
        print('\nリソース一覧:')
        for r in resources.resources:
            print(f'  {r.name} ({r.uri})')

        # リソース読み取り
        res = await client_session.read_resource('memo://sample')
        print('\nメモ内容:', res.contents[0].text)

asyncio.run(run_client())


## カスタマイズのヒント

FastMCPでは関数を書くだけでツールが追加できます。

```python
@mcp.tool()
def my_tool(input: str) -> str:
    '''ツールの説明をdocstringに書く'''
    return f'結果: {input}'
```

| 機能 | デコレータ |
|---|---|
| ツール | `@mcp.tool()` |
| リソース | `@mcp.resource('uri://path')` |
| プロンプト | `@mcp.prompt()` |

## 参考リンク
- [MCP公式ドキュメント](https://modelcontextprotocol.io)
- [mcp Python SDK](https://github.com/modelcontextprotocol/python-sdk)
